# 02 - Extraction Experiments

Test LLM extraction capabilities across different models and document types.

**Models:**
- Claude 3.5 Sonnet (API)
- Llama 3.2 Vision (local via Ollama)
- Tesseract (baseline)

**Metrics:**
- Field accuracy
- JSON validity
- Hallucination rate
- Latency

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.extractors import ClaudeExtractor, OllamaExtractor, TesseractBaseline
from src.evaluators import ExtractionEvaluator, aggregate_results
from src.utils import ExperimentLogger, save_results

In [ ]:
# Initialize
loader = DataLoader("../data")
evaluator = ExtractionEvaluator(fuzzy_threshold=80.0)
logger = ExperimentLogger(log_dir="../results")

print(f"Experiment ID: {logger.experiment_id}")

## Initialize Extractors

In [ ]:
# Initialize extractors
extractors = {}

# Claude (API)
try:
    claude = ClaudeExtractor(model="claude-sonnet-4-20250514")
    extractors["claude"] = claude
    print("✓ Claude extractor ready")
except Exception as e:
    print(f"✗ Claude not available: {e}")

# Ollama (local) - requires Ollama running with vision model
try:
    ollama = OllamaExtractor(model="llama3.2-vision")
    extractors["llama"] = ollama
    print("✓ Ollama extractor ready")
except Exception as e:
    print(f"✗ Ollama not available: {e}")

# Tesseract (baseline)
try:
    tesseract = TesseractBaseline()
    extractors["tesseract"] = tesseract
    print("✓ Tesseract baseline ready")
except Exception as e:
    print(f"✗ Tesseract not available: {e}")

print(f"\nActive extractors: {list(extractors.keys())}")

## Load Test Documents

In [ ]:
# Load documents
test_docs = []

# Try FUNSD first
try:
    funsd = list(loader.load_funsd(split="test"))[:20]  # Limit for quick testing
    test_docs.extend(funsd)
    print(f"Loaded {len(funsd)} FUNSD documents")
except FileNotFoundError:
    print("FUNSD not available")

# Try SROIE
try:
    sroie = list(loader.load_sroie(split="test"))[:20]
    test_docs.extend(sroie)
    print(f"Loaded {len(sroie)} SROIE documents")
except FileNotFoundError:
    print("SROIE not available")

# Fall back to samples
if not test_docs:
    samples = list(loader.load_samples())
    test_docs.extend(samples)
    print(f"Using {len(samples)} sample documents")

print(f"\nTotal test documents: {len(test_docs)}")

## Run Extraction Experiment

In [ ]:
# Run extractions
all_results = []

for extractor_name, extractor in extractors.items():
    print(f"\n{'='*50}")
    print(f"Running: {extractor_name}")
    print(f"{'='*50}")
    
    for doc in tqdm(test_docs, desc=extractor_name):
        # Extract
        result = extractor.extract(doc)
        
        # Evaluate if ground truth available
        if doc.ground_truth:
            eval_result = evaluator.evaluate(
                result.parsed_data,
                doc.ground_truth,
                doc.id,
                extractor_name
            )
        else:
            eval_result = None
        
        # Store
        all_results.append({
            "document_id": doc.id,
            "dataset": doc.dataset,
            "model": extractor_name,
            "extraction": result.to_dict(),
            "evaluation": eval_result.to_dict() if eval_result else None,
        })
        
        logger.add_result(all_results[-1])

print(f"\n\nCompleted {len(all_results)} extractions")

## Analyze Results

In [ ]:
# Convert to DataFrame
rows = []
for r in all_results:
    if r["evaluation"]:
        rows.append({
            "document_id": r["document_id"],
            "dataset": r["dataset"],
            "model": r["model"],
            "field_accuracy": r["evaluation"]["field_accuracy"],
            "fuzzy_score": r["evaluation"]["fuzzy_score"],
            "cer": r["evaluation"]["cer"],
            "wer": r["evaluation"]["wer"],
            "json_valid": r["evaluation"]["json_valid"],
            "hallucinations": r["evaluation"]["hallucinations"],
            "latency_ms": r["extraction"]["latency_ms"],
        })

df = pd.DataFrame(rows)
df.head()

In [ ]:
# Summary by model
if not df.empty:
    summary = df.groupby("model").agg({
        "field_accuracy": "mean",
        "fuzzy_score": "mean",
        "cer": "mean",
        "json_valid": "mean",
        "hallucinations": "sum",
        "latency_ms": "mean",
    }).round(3)
    
    summary.columns = ["Avg Field Accuracy", "Avg Fuzzy Score", "Avg CER", 
                       "JSON Valid Rate", "Total Hallucinations", "Avg Latency (ms)"]
    display(summary)

In [ ]:
# Visualize: Accuracy by Model
if not df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Field Accuracy
    sns.boxplot(data=df, x="model", y="field_accuracy", ax=axes[0])
    axes[0].set_title("Field Accuracy by Model")
    axes[0].set_ylim(0, 1)
    
    # Latency
    sns.boxplot(data=df, x="model", y="latency_ms", ax=axes[1])
    axes[1].set_title("Latency by Model (ms)")
    
    # Hallucinations
    halluc_by_model = df.groupby("model")["hallucinations"].sum()
    halluc_by_model.plot(kind="bar", ax=axes[2])
    axes[2].set_title("Total Hallucinations by Model")
    
    plt.tight_layout()
    plt.savefig("../results/model_comparison.png", dpi=150)
    plt.show()

In [ ]:
# Accuracy by dataset
if not df.empty and df["dataset"].nunique() > 1:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    pivot = df.pivot_table(
        values="field_accuracy",
        index="model",
        columns="dataset",
        aggfunc="mean"
    )
    
    pivot.plot(kind="bar", ax=ax)
    ax.set_title("Field Accuracy by Model and Dataset")
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1)
    ax.legend(title="Dataset")
    
    plt.tight_layout()
    plt.savefig("../results/dataset_comparison.png", dpi=150)
    plt.show()

## Save Results

In [ ]:
# Save all results
logger.save()

# Save summary
if not df.empty:
    df.to_csv(f"../results/{logger.experiment_id}/summary.csv", index=False)
    print(f"Results saved to: results/{logger.experiment_id}/")

## Example Extractions

Show some specific extraction examples for the article.

In [ ]:
# Show example extraction
if all_results:
    example = all_results[0]
    
    print("=" * 50)
    print(f"Document: {example['document_id']}")
    print(f"Model: {example['model']}")
    print("=" * 50)
    
    print("\n--- Extracted Data ---")
    if example['extraction']['parsed_data']:
        print(json.dumps(example['extraction']['parsed_data'], indent=2)[:1000])
    else:
        print("No structured data extracted")
    
    if example['evaluation']:
        print("\n--- Evaluation ---")
        print(f"Field Accuracy: {example['evaluation']['field_accuracy']:.2%}")
        print(f"Hallucinations: {example['evaluation']['hallucinations']}")

## Next Steps

1. Test chaos tolerance: `03_chaos_gradient_test.ipynb`
2. Deep analysis: `04_results_analysis.ipynb`